# TP : Apprentissage Supervisé - Régression (exercice)
## Dataset : California Housing (sklearn)

**Algorithmes couverts :**

| Algo | Famille | Caractéristique |
|---|---|---|
| Régression Linéaire (Ridge/Lasso) | Linéaire | Interprétable, régularisable |
| KNN | Instance-based | Simple, sensible à l'échelle |
| Decision Tree | Arbre | Interprétable, sujet à l'overfitting |
| Random Forest | Ensemble (bagging) | Robuste, peu de réglage |
| XGBoost | Ensemble (boosting) | Souvent le plus performant |

**Datamap (dictionnaire des données) :** 8 variables socio-démographiques par quartier (bloc de recensement californien).

| Colonne | Description |
|---|---|
| MedInc | Revenu médian du quartier (en dizaines de milliers de $) |
| HouseAge | Âge médian des logements |
| AveRooms | Nombre moyen de pièces par logement |
| AveBedrms | Nombre moyen de chambres par logement |
| Population | Population du quartier |
| AveOccup | Nombre moyen d'occupants par logement |
| Latitude | Latitude géographique |
| Longitude | Longitude géographique |

**Cible :** `target` : valeur médiane des logements du quartier (en centaines de milliers de $).

> **Version exercice** : les cellules marquées `# TODO` sont à compléter. Le reste (imports, chargement des données, affichages) est déjà fourni.
> Installe les dépendances une seule fois avec `pip install -r requirements.txt` depuis `cours_ml/todo/` (voir le README de ce dossier). Ce TP s'inspire de `cours_ml/02_supervise/tp_regression.ipynb` (même méthode, données différentes et plus volumineuses : 20640 quartiers californiens contre 442 patients).

---
## 0. Imports & configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42


---
## 1. Chargement & exploration

In [ ]:
# California Housing : 20640 quartiers, 8 variables socio-demographiques -> valeur mediane des logements
data = fetch_california_housing()
X, y = data.data, data.target
feature_names = list(data.feature_names)

df_raw = pd.DataFrame(X, columns=feature_names)
df_raw['target'] = y

print(f"Shape : {df_raw.shape}")
print("target = valeur mediane des logements (en centaines de milliers de $)")
df_raw.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y, bins=30, color='steelblue', edgecolor='k')
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f'Moyenne ({y.mean():.2f})')
axes[0].set_title('Distribution de la cible (valeur médiane du logement)')
axes[0].legend()

axes[1].scatter(df_raw['Longitude'], df_raw['Latitude'], c=y, cmap='viridis', alpha=0.3, s=8)
axes[1].set_title('Valeur médiane par position géographique')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

plt.tight_layout()
plt.show()


In [ ]:
corr_target = df_raw.corr()['target'].drop('target')
top_features = corr_target.abs().sort_values(ascending=False).head(4).index.tolist()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, top_features):
    ax.scatter(df_raw[col], y, alpha=0.15, s=8, color='steelblue')
    ax.set_xlabel(col)
    ax.set_ylabel('target')
    ax.set_title(f'r = {corr_target[col]:.2f}')
plt.tight_layout()
plt.show()


---
## 2. Prétraitement & split

In [ ]:
# TODO : separer train/test (80/20), normaliser avec StandardScaler, et preparer la cross-validation (KFold 5)
# Indice : train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = ...

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"Train : {X_train.shape[0]} | Test : {X_test.shape[0]}")


---
## 3. Régression Linéaire

Modèle de la forme $\hat{y} = w^T x + b$, minimisant la somme des carrés des résidus (OLS).
On compare la version basique avec **Ridge** (régularisation L2) et **Lasso** (L1, sélection de variables).

In [ ]:
# Régression linéaire OLS
lr = LinearRegression()
lr.fit(X_train_sc, y_train)

coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': lr.coef_})
coef_df = coef_df.sort_values('coefficient', key=abs, ascending=False)

plt.figure(figsize=(8, 4))
colors = ['crimson' if v < 0 else 'steelblue' for v in coef_df['coefficient']]
plt.bar(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='k')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Régression Linéaire : Coefficients')
plt.ylabel('Coefficient')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Optimisation Ridge et Lasso
alpha_values = np.logspace(-3, 3, 20)
cv_ridge, cv_lasso = [], []

for alpha in alpha_values:
    # TODO : mesurer le R2 en cross-validation pour Ridge(alpha=alpha) et Lasso(alpha=alpha, max_iter=5000)
    # Indice : cross_val_score(..., X_train_sc, y_train, cv=cv, scoring='r2').mean()
    r = ...
    l = ...
    cv_ridge.append(r)
    cv_lasso.append(l)

best_alpha_ridge = alpha_values[np.argmax(cv_ridge)]
best_alpha_lasso = alpha_values[np.argmax(cv_lasso)]

plt.figure(figsize=(9, 4))
plt.semilogx(alpha_values, cv_ridge, 'b-o', label='Ridge')
plt.semilogx(alpha_values, cv_lasso, 'r-s', label='Lasso')
plt.axvline(best_alpha_ridge, color='b', linestyle='--', alpha=0.5)
plt.axvline(best_alpha_lasso, color='r', linestyle='--', alpha=0.5)
plt.xlabel('Alpha (régularisation)')
plt.ylabel('R² (CV 5-fold)')
plt.title('Ridge vs Lasso : Optimisation de alpha')
plt.legend()
plt.tight_layout()
plt.show()

# TODO : entrainer la meilleure regression lineaire (Ridge avec alpha=best_alpha_ridge)
lr_best = ...
print(f"Ridge optimal : alpha={best_alpha_ridge:.4f} | R² CV = {max(cv_ridge):.4f}")


---
## 4. KNN Régression

Prédit la valeur d'un point comme la **moyenne des k voisins les plus proches**.

In [ ]:
k_values = range(1, 31)
cv_knn, cv_knn_train = [], []

for k in k_values:
    # TODO : entrainer un KNeighborsRegressor(n_neighbors=k), stocker le R2 train et le R2 en cross-validation
    knn = ...
    knn.fit(X_train_sc, y_train)
    cv_knn_train.append(...)
    scores = ...
    cv_knn.append(scores.mean())

best_k = list(k_values)[np.argmax(cv_knn)]

plt.figure(figsize=(9, 4))
plt.plot(k_values, cv_knn_train, 'b--o', alpha=0.7, label='R² train')
plt.plot(k_values, cv_knn, 'rs-', label='R² CV')
plt.axvline(best_k, color='green', linestyle='--', label=f'k optimal = {best_k}')
plt.xlabel('k (voisins)')
plt.ylabel('R²')
plt.title('KNN Régression : Overfitting vs k')
plt.legend()
plt.tight_layout()
plt.show()

# TODO : entrainer le KNN final avec n_neighbors=best_k
knn_best = ...
print(f"k optimal : {best_k} | R² CV : {max(cv_knn):.4f}")


---
## 5. Decision Tree Régression

In [ ]:
depth_values = range(1, 21)
cv_dt, cv_dt_train = [], []

for d in depth_values:
    # TODO : entrainer un DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE), stocker R2 train et R2 CV
    dt = ...
    dt.fit(X_train_sc, y_train)
    cv_dt_train.append(...)
    scores = ...
    cv_dt.append(scores.mean())

best_depth = list(depth_values)[np.argmax(cv_dt)]

plt.figure(figsize=(9, 4))
plt.plot(depth_values, cv_dt_train, 'b--o', alpha=0.7, label='R² train')
plt.plot(depth_values, cv_dt, 'rs-', label='R² CV')
plt.axvline(best_depth, color='green', linestyle='--', label=f'Profondeur optimale = {best_depth}')
plt.xlabel('max_depth')
plt.ylabel('R²')
plt.title('Decision Tree Régression : Overfitting vs Profondeur')
plt.legend()
plt.tight_layout()
plt.show()

# TODO : entrainer l'arbre final avec max_depth=best_depth
dt_best = ...
print(f"Profondeur optimale : {best_depth} | R² CV : {max(cv_dt):.4f}")


---
## 6. Random Forest Régression

In [ ]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'max_features': ['sqrt', 0.5],
}

# TODO : chercher les meilleurs hyperparametres avec GridSearchCV
# Indice : GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE), param_grid_rf, cv=cv, scoring='r2', n_jobs=-1)
rf = ...
gs_rf = ...
gs_rf.fit(X_train_sc, y_train)

print(f"Meilleurs paramètres RF : {gs_rf.best_params_}")
print(f"R² CV                  : {gs_rf.best_score_:.4f}")

rf_best = gs_rf.best_estimator_


In [ ]:
importances = pd.Series(rf_best.feature_importances_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
importances.plot(kind='bar', color='steelblue', edgecolor='k')
plt.title('Random Forest : Feature importances')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


---
## 7. XGBoost Régression

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
}

# TODO : chercher les meilleurs hyperparametres avec GridSearchCV
# Indice : GridSearchCV(XGBRegressor(random_state=RANDOM_STATE, verbosity=0), param_grid_xgb, cv=cv, scoring='r2', n_jobs=-1)
xgb = ...
gs_xgb = ...
gs_xgb.fit(X_train_sc, y_train)

print(f"Meilleurs paramètres XGB : {gs_xgb.best_params_}")
print(f"R² CV                   : {gs_xgb.best_score_:.4f}")

xgb_best = gs_xgb.best_estimator_


---
## 8. Évaluation & comparaison

### 8.1 Métriques sur le jeu de test

In [ ]:
models = {
    'Linear Regression (Ridge)': lr_best,
    'KNN': knn_best,
    'Decision Tree': dt_best,
    'Random Forest': rf_best,
    'XGBoost': xgb_best,
}

results = []
for name, model in models.items():
    # TODO : predire y_pred sur X_test_sc puis calculer MAE, RMSE (fonction rmse deja definie) et R2
    y_pred = ...
    results.append({
        'Modèle': name,
        'MAE': ...,
        'RMSE': ...,
        'R²': ...,
    })

df_results = pd.DataFrame(results).sort_values('R²', ascending=False)
df_results.round(4)


### 8.2 Visualisation des métriques

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['steelblue', 'darkorange', 'seagreen', 'crimson', 'purple']
model_names = df_results['Modèle'].tolist()

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R²']):
    ax.bar(model_names, df_results[metric], color=colors, edgecolor='k')
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


### 8.3 Prédictions vs valeurs réelles

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (name, model), color in zip(axes, models.items(), colors):
    y_pred = model.predict(X_test_sc)
    ax.scatter(y_test, y_pred, alpha=0.15, s=8, color=color)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=1)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Réel')
    ax.set_ylabel('Prédit')

plt.tight_layout()
plt.show()


### 8.4 Analyse des résidus du meilleur modèle

In [ ]:
best_model_name = df_results.iloc[0]['Modèle']
best_model = models[best_model_name]
# TODO : predire y_pred_best sur X_test_sc et calculer les residus (y_test - y_pred_best)
y_pred_best = ...
residuals = ...

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(y_pred_best, residuals, alpha=0.15, edgecolors='k', linewidths=0.1, s=15)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Valeurs prédites')
axes[0].set_ylabel('Résidus')
axes[0].set_title('Résidus vs Prédictions')

axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='k')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Résidus')
axes[1].set_title('Distribution des résidus')

from scipy import stats
stats.probplot(residuals, plot=axes[2])
axes[2].set_title('Q-Q Plot des résidus')

plt.suptitle(f'Analyse des résidus : {best_model_name}', fontsize=11)
plt.tight_layout()
plt.show()

print(f"Meilleur modèle : {best_model_name}")
print(f"  MAE  : {mean_absolute_error(y_test, y_pred_best):.2f}")
print(f"  RMSE : {rmse(y_test, y_pred_best):.2f}")
print(f"  R²   : {r2_score(y_test, y_pred_best):.4f}")


---
## 9. Courbes d'apprentissage (Learning Curves)

Montrent comment la performance évolue avec la taille du dataset d'entraînement : diagnostique le biais/variance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models_lc = {
    'Ridge': Ridge(alpha=best_alpha_ridge),
    'Random Forest': RandomForestRegressor(**gs_rf.best_params_, random_state=RANDOM_STATE),
    'XGBoost': XGBRegressor(**gs_xgb.best_params_, random_state=RANDOM_STATE, verbosity=0),
}

train_sizes = np.linspace(0.1, 1.0, 8)

for ax, (name, model) in zip(axes, models_lc.items()):
    # TODO : calculer les courbes d'apprentissage avec learning_curve
    # Indice : learning_curve(model, X_train_sc, y_train, train_sizes=train_sizes, cv=cv, scoring='r2', n_jobs=-1)
    sizes, train_sc_, val_sc_ = ...
    ax.plot(sizes, train_sc_.mean(1), 'b-o', label='Train')
    ax.fill_between(sizes, train_sc_.mean(1)-train_sc_.std(1), train_sc_.mean(1)+train_sc_.std(1), alpha=0.1, color='b')
    ax.plot(sizes, val_sc_.mean(1), 'r-s', label='Validation')
    ax.fill_between(sizes, val_sc_.mean(1)-val_sc_.std(1), val_sc_.mean(1)+val_sc_.std(1), alpha=0.1, color='r')
    ax.set_xlabel("Taille du dataset d'entraînement")
    ax.set_ylabel('R²')
    ax.set_title(f'Learning Curve : {name}')
    ax.legend()

plt.tight_layout()
plt.show()


---
## 10. Explicabilité avec SHAP

Un modèle boîte noire (Random Forest, XGBoost) est précis mais difficile à justifier auprès du métier. Les **valeurs de Shapley** (SHAP) attribuent à chaque variable sa contribution à une prédiction, sans changer le modèle :
$$\text{prédiction} = \text{valeur de base} + \sum_{i} \text{valeur de Shapley}(x_i)$$

On illustre la méthode sur XGBoost.

In [ ]:
import shap

# TODO : creer un TreeExplainer sur xgb_best, puis calculer les shap_values sur X_test_sc
# Indice : explainer = shap.TreeExplainer(xgb_best) ; shap_values = explainer.shap_values(X_test_sc)
explainer = ...
shap_values = ...

shap.summary_plot(shap_values, X_test_sc, feature_names=feature_names)

**Lecture du graphique (summary plot) :** chaque point est un quartier du jeu de test. La position horizontale indique l'impact sur la valeur médiane prédite (vers la droite : augmente la prédiction, vers la gauche : la diminue), et la couleur la valeur de la feature (rouge = valeur élevée, bleu = valeur faible). Les variables sont triées par impact moyen absolu (importance globale).

In [ ]:
# TODO : expliquer la premiere observation du jeu de test et tracer un waterfall plot
# Indice : explanation = explainer(X_test_sc[:1]) ; shap.plots.waterfall(explanation[0])
...

print(f"Valeur réelle : {y_test[0]:.2f} | Valeur prédite : {xgb_best.predict(X_test_sc[:1])[0]:.2f}")

---
## 11. Choisir le modèle final : erreur, performance et explicabilité

Le modèle final ne se choisit pas seulement sur la métrique de performance : trois critères entrent en jeu :
1. **Minimiser l'erreur** (ici : RMSE)
2. **Maximiser la performance** (ici : R², variance expliquée)
3. **Maximiser l'explicabilité** (nativement interprétable, ou via SHAP pour une boîte noire)

In [ ]:
explicabilite = {
    'Linear Regression (Ridge)': 'Haute (coefficients directs)',
    'KNN': 'Faible (pas de règle ni de coefficient)',
    'Decision Tree': 'Très haute (règles lisibles)',
    'Random Forest': 'Moyenne (boîte noire, expliquée via SHAP)',
    'XGBoost': 'Moyenne (boîte noire, expliquée via SHAP)',
}

# TODO : construire df_choix a partir de df_results, avec une colonne 'Explicabilite'
# (via explicabilite.map sur la colonne 'Modèle')
df_choix = ...
df_choix = df_choix[['Modèle', 'RMSE', 'R²', 'Explicabilité']]
df_choix.round(4)

In [ ]:
# TODO : determiner le modele final (meilleure ligne de df_results par R2, deja trie) et son modele associe,
# puis calculer l'ecart de R2 avec la regression lineaire (modele le plus interpretable)
best_final_name = ...
best_final_model = ...
r2_gap = ...

print(f"Modèle final retenu : {best_final_name}")
print(f"  RMSE          : {df_results.iloc[0]['RMSE']:.4f}")
print(f"  R²            : {df_results.iloc[0]['R²']:.4f}")
print(f"  Explicabilité : {explicabilite[best_final_name]}")
print(f"  Écart de R² avec Linear Regression (Ridge) : {r2_gap:.4f}")

---
## 12. Stocker le modèle final

In [ ]:
import joblib
import os

os.makedirs('modeles', exist_ok=True)
# TODO : sauvegarder best_final_model ET scaler avec joblib
# (tous les modeles de ce TP sont entraines sur X_train_sc, pas de Pipeline : le scaler doit etre sauvegarde a part)
...
...

print(f"Modèle sauvegardé : modeles/best_model_regression.pkl ({best_final_name})")
print("Scaler sauvegardé : modeles/scaler_regression.pkl")

---
## 13. Inférence simple, sans API

In [ ]:
# Nouvelles données à prédire (ici, un échantillon du jeu de test, pour l'exemple :
# en production ces lignes viendraient d'une nouvelle source, pas du jeu de test)
nouvelles_donnees = pd.DataFrame(X_test[:10], columns=feature_names)

# TODO : charger le modele et le scaler sauvegardes, standardiser nouvelles_donnees, puis predire
model_charge = ...
scaler_charge = ...
X_new_sc = ...
predictions = ...

nouvelles_donnees['prediction_valeur_mediane'] = predictions.round(2)
nouvelles_donnees.to_csv('predictions_regression.csv', index=False)

print(f"Prédictions sauvegardées : predictions_regression.csv ({len(nouvelles_donnees)} lignes)")
nouvelles_donnees[['prediction_valeur_mediane']]

---
## 14. Conclusion

| Critère | Lin. Reg. | KNN | Decision Tree | Random Forest | XGBoost |
|---|---|---|---|---|---|
| Interprétabilité | ★★★★★ | ★★ | ★★★★ | ★★ | ★★ |
| Performance typique | ★★★ | ★★ | ★★★ | ★★★★ | ★★★★★ |
| Sensible à l'échelle | Oui | Oui | Non | Non | Non |
| Passage à l'échelle (20k lignes) | Excellent | Moyen (inférence lente) | Excellent | Bon | Bon |

**À retenir :** avec 20640 lignes, KNN devient nettement plus lent à l'inférence que les modèles à base d'arbres : la taille des données influence aussi le choix pratique de l'algorithme, pas seulement sa performance.

---
## Session à rendre

Cette section est à compléter et à rendre à l'issue du TP. Réponds à chaque question dans la
cellule *Réponse* juste en dessous, à partir des résultats que **tu as toi-même obtenus** en
réalisant ce notebook (il n'y a pas de réponse générique valable pour tout le monde : les valeurs
numériques, choix d'hyperparamètres et graphiques dépendent de ton exécution).

**Q1.** Quel alpha as-tu retenu pour Ridge et pour Lasso ? Que se passe-t-il sur les coefficients quand alpha augmente ?

*Réponse :*

_(à compléter)_

**Q2.** Quel k as-tu choisi pour KNN en régression ?

*Réponse :*

_(à compléter)_

**Q3.** Quelle profondeur pour l'arbre de décision, et quel est l'effet sur le compromis biais/variance ?

*Réponse :*

_(à compléter)_

**Q4.** Quels hyperparamètres GridSearchCV a-t-il retenus pour Random Forest et XGBoost ?

*Réponse :*

_(à compléter)_

**Q5.** D'après ton tableau récapitulatif, quel modèle minimise le mieux le RMSE (ou maximise le R²) ? Est-ce cohérent avec ce que tu attendais ?

*Réponse :*

_(à compléter)_

**Q6.** Que montrent les résidus de ton meilleur modèle : y a-t-il un biais systématique (sous- ou sur-estimation) pour certaines gammes de prix ?

*Réponse :*

_(à compléter)_

**Q7.** Que montre la learning curve : le modèle bénéficierait-il de plus de données, ou est-il plutôt limité par sa capacité (biais vs variance) ?

*Réponse :*

_(à compléter)_

**Q8.** Sur le summary plot SHAP de XGBoost, quelles sont les 2-3 variables qui ont le plus d'impact sur la valeur médiane prédite ?

*Réponse :*

_(à compléter)_

**Q9.** Quel modèle final as-tu obtenu après arbitrage erreur/performance/explicabilité (section 11) ? Le gain de performance du modèle boîte noire sur la régression linéaire est-il marginal ou net dans ton cas ?

*Réponse :*

_(à compléter)_

**Q10.** Sur les 10 nouvelles prédictions sauvegardées dans `predictions_regression.csv`, quelle est la valeur médiane prédite la plus haute et la plus basse ?

*Réponse :*

_(à compléter)_